# Laboratorio de Apache Hive (MetaHive)

Asegúrate de haber inicializado primero los clústeres corriendo tanto `hadoop_infra.ipynb` como `hive_infra.ipynb` antes de ejecutar este laboratorio.


In [1]:
HIVE_DATADIR = "/user/hive/warehouse"

## 1. Preparar un dataset en HDFS

Vamos a simular un archivo de ventas y lo subiremos a HDFS. Hive necesita que los datos estén alojados en HDFS para poder leerlos transparentemente.


In [2]:
import csv
import random
import os

dataset_filename = "sales.csv"
with open(dataset_filename, "w", newline="") as f:
    writer = csv.writer(f)
    # Hive puede ignorar headers fácilmente, pero para simplificar la primera tabla, lo generaremos sin cabecera.
    # Formato esperado: id, producto, categoria, monto, pais
    for i in range(1, 50001):
        writer.writerow(
            [
                i,
                f"Producto_{random.randint(1, 20)}",
                random.choice(["Electronica", "Hogar", "Deportes"]),
                round(random.uniform(10.0, 500.0), 2),
                random.choice(["MX", "US", "CO", "ES"]),
            ]
        )
print(f"¡Dataset '{dataset_filename}' generado localmente con 500,000 filas!")

¡Dataset 'sales.csv' generado localmente con 500,000 filas!


In [3]:
# Pasamos el archivo del host al contenedor del namenode temporalmente, y luego lo consagramos en HDFS
!docker exec namenode hdfs dfs -mkdir -p {HIVE_DATADIR}/warehouse/unmanaged/sales
!docker cp sales.csv namenode:/tmp/sales.csv
!docker exec namenode hdfs dfs -put /tmp/sales.csv {HIVE_DATADIR}/warehouse/unmanaged/sales/

## 2. Conexión a Hive via Beeline

Apache Hive procesa consultas SQL (conocido como HQL) y el Metastore convierte esas declaraciones en grafos de ejecución de MapReduce o Tez. Nos conectaremos al motor vía `beeline`.


In [4]:
# Verificando conexión basica al servidor.
!docker exec -e TERM=dumb hive-server2 beeline -u "jdbc:hive2://localhost:10000" -n hive -e "SHOW DATABASES;" 

+----------------+
| database_name  |
+----------------+
| default        |
+----------------+


SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/tez/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/tez/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.c

## 3. Creación de Base de Datos y una EXTERNAL TABLE

Las tablas **Externas** en Hive mapean un esquema SQL encima de un directorio puro de HDFS. Si ejecutas un `DROP TABLE` sobre una tabla externa, Hive tira su esquema a la basura pero ¡respeta tus valiosos archivos en el sistema HDFS!


In [5]:
# Nota: beeline -f tiene un bug en Hive 4.0.0 con JLine cuando no hay TTY.
# Usamos multiples llamadas -e (una por sentencia) que si funcionan sin TTY.

!docker exec -e TERM=dumb hive-server2 beeline -u "jdbc:hive2://localhost:10000/" -n hive -e "DROP DATABASE IF EXISTS lab_db CASCADE"

!docker exec -e TERM=dumb hive-server2 beeline -u "jdbc:hive2://localhost:10000/" -n hive -e "CREATE DATABASE IF NOT EXISTS lab_db"

create_table_sql = "".join([
    "CREATE EXTERNAL TABLE IF NOT EXISTS sales (",
    " id INT, producto STRING, categoria STRING, monto DOUBLE, pais STRING)",
    " ROW FORMAT DELIMITED FIELDS TERMINATED BY ','",
    " STORED AS TEXTFILE",
    f" LOCATION 'hdfs://namenode.mavasbel.vpn.itam.mx:8020{HIVE_DATADIR}/warehouse/unmanaged/sales'"
])
!docker exec -e TERM=dumb hive-server2 beeline -u "jdbc:hive2://localhost:10000/lab_db" -n hive -e "{create_table_sql}"

!docker exec -e TERM=dumb hive-server2 beeline -u "jdbc:hive2://localhost:10000/lab_db" -n hive -e "SHOW TABLES"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/tez/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/tez/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.c

+-----------+
| tab_name  |
+-----------+
| sales     |
+-----------+


SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/tez/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/opt/hive/lib/log4j-slf4j-impl-2.18.0.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/tez/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/opt/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.c

## 4. Consulta Directa (Fetch Task)

Un `SELECT *` no requiere cálculo distribuido agresivo. Hive lo identifica y solo pide a HDFS el volcado del archivo como un simple visor de tabla.


In [ ]:
!docker exec -e TERM=dumb hive-server2 beeline -u "jdbc:hive2://localhost:10000/lab_db" -n hive -e "SELECT * FROM sales LIMIT 5"

## 5. Consultas con Agregación (Activando procesamiento)

Al usar funciones complejas o `GROUP BY`, notarás cómo Beeline comienza a imprimir los logs de MapReduce en pantalla, porque detrás de la familiaridad del SQL, Hive acaba de despachar un Job al ResourceManager.


In [ ]:
# Esta query usa un self-JOIN que multiplica el dataset (~500k x 500k/paises)
# Forzamos el motor MapReduce para que puedas ver el job en el YARN UI (localhost:8088)
# Abre http://localhost:8088 en tu navegador mientras esto corre!

# !docker exec -e TERM=dumb hive-server2 beeline -u "jdbc:hive2://localhost:10000/lab_db" -n hive -e "SET hive.execution.engine=mr"
!docker exec -e TERM=dumb hive-server2 beeline -u "jdbc:hive2://localhost:10000/lab_db" -n hive -e "SELECT a.categoria, b.pais, COUNT(*) as combinaciones, ROUND(AVG(a.monto + b.monto), 2) as ticket_combinado_promedio FROM sales a JOIN sales b ON a.pais = b.pais WHERE a.id != b.id GROUP BY a.categoria, b.pais ORDER BY combinaciones DESC"

## 6. Manejo Predictivo: Particionamiento (Managed Tables)

El poder de MetaHive en Big Data surge de saltar la lectura física de petabytes enteros de datos. Transformaremos nuestra tabla plana a una **Partitioned Table**. Físicamente agrupará la data por país, lo que nos permitiría consultar "Ventas en MX" en microsegundos porque ¡ignoraría físicamente las carpetas de los demás países!


In [ ]:
# INSERT OVERWRITE activa un MapReduce job masivo para particionar la data.
!docker exec -e TERM=dumb hive-server2 beeline -u "jdbc:hive2://localhost:10000/lab_db" -n hive -e "SET hive.exec.dynamic.partition=true; SET hive.exec.dynamic.partition.mode=nonstrict"
!docker exec -e TERM=dumb hive-server2 beeline -u "jdbc:hive2://localhost:10000/lab_db" -n hive -e "CREATE TABLE IF NOT EXISTS sales_partitioned (id INT, producto STRING, categoria STRING, monto DOUBLE) PARTITIONED BY (pais STRING) STORED AS SEQUENCEFILE"
!docker exec -e TERM=dumb hive-server2 beeline -u "jdbc:hive2://localhost:10000/lab_db" -n hive -e "INSERT OVERWRITE TABLE sales_partitioned PARTITION(pais) SELECT id, producto, categoria, monto, pais FROM sales"

### Probando la estructura granular subyacente

Las particiones en Hive equivalen mágicamente a sub-directorios dentro de HDFS bajo la bandera `nombre_columna=valor`. ¡Verifiquémoslo nativamente desde el File System distribuido!


In [ ]:
# Se guarda por defecto en el 'warehouse directory' configurado en {HIVE_DATADIR}/warehouse/
!docker exec namenode hdfs dfs -ls {HIVE_DATADIR}/warehouse/external/lab_db.db/sales_partitioned